## NECESSARY IMPORTS 

In [0]:
from pyspark.sql import functions as F 
from pyspark.sql.types import IntegerType , DoubleType 

In [0]:
dbutils.widgets.text("catalog" , "ecommerce_catalog_dev")
catalog = dbutils.widgets.get("catalog")

## String Cleaning

In [0]:
df_bronze = spark.table(f"{catalog}.bronze.raw_orders")
df_str_clean = df_bronze\
    .withColumn("customer_id" , F.trim(F.col("customer_id")))\
    .withColumn("product_name" , F.lower(F.trim(F.col("product_name"))))\
    .withColumn("status" , F.upper(F.trim(F.col("status"))))


## Type Casting

In [0]:
df_casted = df_str_clean\
    .withColumn("order_id" , F.col("order_id").cast(IntegerType()))\
    .withColumn("quantity" , F.col("quantity").cast(IntegerType()))\
    .withColumn("unit_price" , F.col("unit_price").cast(DoubleType()))

## Date Parsing (Handling mixed date formats)

In [0]:
df_dated = df_casted.withColumn(
    "order_date",
    F.coalesce(
        F.try_to_date(F.col("order_date") , "yyyy-MM-dd"),
        F.try_to_date(F.col("order_date") , "MM/dd/yyyy"),
        F.try_to_date(F.col("order_date") , "dd-MM-yyyy")
    )
)

## Data Filtering (Data Quality Checks)

In [0]:
df_filtered = df_dated\
    .filter(F.col("order_id").isNotNull())\
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id")!=""))\
    .filter(F.col("order_date").isNotNull())\
    .filter(F.col("product_name").isNotNull())\
    .filter(F.col("quantity").isNotNull() & (F.col("quantity") > 0))\
    .filter(F.col("unit_price").isNotNull() & (F.col("unit_price") > 0))\
    .filter(F.col("status").isin(["COMPLETED", "PENDING", "CANCELLED"]))

## Deduplication

In [0]:
df_deduped = df_filtered.dropDuplicates(["order_id"])

## Feature Engineering (Calculate total_amount)

In [0]:
df_silver = df_deduped.withColumn("total_amount" , F.round(F.col("quantity") * F.col("unit_price") , 2))

## Write to Silver Table

In [0]:
df_silver.write\
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.silver.silver_table")